# Ch 12　Subgraphs 與模組化

> **本 notebook 完全不需要 API key。**

In [ ]:
!uv pip install -q langgraph

## 12.4　模式二：共享 state，直接把編譯好的 subgraph 當節點

父圖與 subgraph 共用同一組 key 時，連包裝函數都不用——直接 `add_node(subgraph)`。

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class Shared(TypedDict):
    question: str      # 父子共用這兩個 channel（無 reducer，後寫覆蓋）
    answer: str

# subgraph：把「檢索 + 摘要」封裝成可重用的一張圖，讀 question、寫 answer
def retrieve(state): return {"answer": f"關於「{state['question']}」的檢索結果"}
def summarize(state): return {"answer": state["answer"] + "（已摘要）"}
sub = (StateGraph(Shared).add_node("retrieve", retrieve).add_node("summarize", summarize)
       .add_edge(START, "retrieve").add_edge("retrieve", "summarize").add_edge("summarize", END)
       .compile())

# 父圖：直接把編譯好的 subgraph 當成一個節點塞進去（共用 question / answer）
parent = (StateGraph(Shared).add_node("prepare", lambda s: {"question": s["question"].strip()})
          .add_node("rag", sub)                       # ← subgraph 當節點！不必寫包裝函數
          .add_edge(START, "prepare").add_edge("prepare", "rag").add_edge("rag", END)
          .compile())

print(parent.invoke({"question": "  什麼是 LangGraph？  ", "answer": ""}))

## 12.3　模式一：schema 不同，在節點內呼叫 subgraph 並轉換

In [ ]:
from typing_extensions import TypedDict

# subgraph 有自己的 schema（用 bar）
class SubState(TypedDict):
    bar: str
def sub_node(state: SubState): return {"bar": "hi! " + state["bar"]}
subg = (StateGraph(SubState).add_node("sub_node", sub_node)
        .add_edge(START, "sub_node").add_edge("sub_node", END).compile())

# 父圖用不同的 schema（用 foo）→ 節點當「翻譯官」轉換進出
class Parent(TypedDict):
    foo: str
def call_sub(state: Parent):
    out = subg.invoke({"bar": state["foo"]})   # 進去前：foo → bar
    return {"foo": out["bar"]}                 # 出來後：bar → foo

pg = (StateGraph(Parent).add_node("call_sub", call_sub)
      .add_edge(START, "call_sub").add_edge("call_sub", END).compile())
print(pg.invoke({"foo": "Kevin"}))

## 12.5　用 subgraphs=True 看見 subgraph 內部（看 ns）

In [ ]:
# 串流時加 subgraphs=True，事件會帶 namespace（ns）告訴你來自哪一層。
for chunk in parent.stream({"question": "什麼是 LangGraph？", "answer": ""},
                           subgraphs=True, version="v2"):
    if chunk["type"] == "updates":
        layer = "父圖" if chunk["ns"] == () else f"subgraph {chunk['ns']}"
        print(f"[{layer}] {chunk['data']}")